# || NEMO Batch Analysis ||
© Konstantinos Andreadis 2024 (PhD @ Roux Lab & Salbreux Lab at UNIGE, Switzerland)

In [ ]:
%load_ext autoreload
%autoreload 2

# Import custom module_scripts
from automated_scripts import nemo_0_load_img, nemo_1_preprocess_img, nemo_2_mesh_img, nemo_3_project_img, \
    nemo_4_extract_nematic, nemo_5_analyse_nematic, nemo_6_extract_defects, nemo_7_analyse_defects, nemo_8_interlayer_nematic, \
    nemo_morph_curvature, nemo_morph_thickness
from module_scripts import analysis, datahandler

# Import python essentials
from itertools import product
import numpy as np

In [ ]:
# Fill list with full TIF image paths
img_list = [
    '/Users/andreadi/Physbio Dropbox/Konstantinos Andreadis/Academic/Data/!nemo_figure_runs/hydra-foot.tif',
]

In [ ]:
show_figures = False
render = False

# ==== Load image ====
channel_colours = {
    0: ["Greens_r", "Greens", "green"],
    1: ["inferno", "inferno_r", "inferno"],
    2: ["Blues_r", "Blues", "blue"],
    3: ["Reds_r", "Reds", "red"],
}

# ==== Blur & threshold image ====
blur_value = 3
thresh_val = 3

# ==== Mesh image ====
mesh_marchcube_boxsize = 2
taubin_smooth_passband = 0.001
taubin_smooth_iterations = 200
mesh_split_mode = "radial_spherical"

# ==== TASK lISTS ====
proj_tasks = [
    {
        "mesh_name": "sampling_bottom",
        "dist_min": 0.0,
        "dist_max": 12.0,
        "dist_num": 20,
        "proj_mode": "mean",
        "flip_normals": False
    },
    {
        "mesh_name": "outer_mesh_smooth_subset",
        "dist_min": 0.0,
        "dist_max": 12.0,
        "dist_num": 20,
        "proj_mode": "mean",
        "flip_normals": False
    }
]

inter_layer_order_tasks = [
    {
        "layer_name_1": "inner_mesh_smooth_subset_proj_0.0_to_12.0_um_mean",
        "patch_label_1": "r-50.0um",
        "layer_name_2": "outer_mesh_smooth_subset_proj_0.0_to_12.0_um_mean",
        "patch_label_2": "r-50.0um"
    }
]

thickness_calc_tasks = [
    {
        "mesh_1_name": "inner_mesh_smooth_subset",
        "mesh_2_name": "outer_mesh_smooth_subset"
    }
]

curv_calc_tasks = [
    {
        "mesh_name": "full_mesh_smooth_subset",
        "flip_normals": False,
        "radius": 20
    },
]

# ==== CONSTANT PARAMETERS ====
proj_scale_down_mesh = True

nematic_patch_mode = "radius"
nematic_patch_size = 30
nematic_compute_num = 7000
nematic_normal_validity_k = 20
nematic_normal_validity_thresh = 0.99
nematic_grid_n = 30

nematic_avg_mode = "radius"
nematic_avg_size = 50.0

defect_dist_cutoff_localisation = 50
defect_max_candidates = 20

defect_topcurv_radius = 50
defect_topcurv_interpk = 10
defect_topcharge_mode = "radius"
defect_topcharge_size = 25.0
defect_pol_mode = "radius"
defect_pol_size = 50.0

thickness_num_samples = 5000
thickness_interp_k = 10

curv_num_samples = 2000
curv_interp_k = 10

# ==== MAIN RUN ====
for i, path in enumerate(img_list):
    print("||||||||||", len(img_list) - i, "left !")
    progress = 100 * np.round(i / max(1, len(img_list)), 2)
    print(f"=========== [Progress {progress}%] {path} ===========")

    dims = analysis.load_img_dimensions(path)
    num_timepoints = dims["T"]
    num_channels = dims["C"]

    for t_select, c_select in product(range(num_timepoints), range(num_channels)):
        if t_select < 80:
            continue
        slice_col, max_col, ren_col = channel_colours.get(c_select, ["Greens_r", "Greens", "green"])

        # ==== Load image ====
        img_load = nemo_0_load_img.main(
            img_path=path, t_select=t_select, c_select=c_select,
            img_slice_colour=slice_col, img_maxproj_colour=max_col, render_colour=ren_col,
            render=render, show_figures=show_figures
        )
        if img_load is None:
            print(f"[!] Issue loading {path} T={t_select} C={c_select} !")
            continue
        img_raw, img_dim, img_scale, img_unit = img_load
        resdata_dir, resfig_dir = datahandler.create_resdirs(path, ct_label=f"t={t_select}_c={c_select}")

        # ==== Blur & threshold image ====
        img_blur, img_thresh = nemo_1_preprocess_img.main(
            img_path=path, t_select=t_select, c_select=c_select,
            img_blur_val=blur_value, img_thresh_val=thresh_val,
            show_figures=show_figures, render=render
        )

        # ==== Mesh image ====
        raw_meshes, smooth_meshes, smooth_subset_meshes = nemo_2_mesh_img.main(
            img_path=path, t_select=t_select, c_select=c_select,
            box_size=mesh_marchcube_boxsize, taubin_smooth_passband=taubin_smooth_passband,
            taubin_smooth_iterations=taubin_smooth_iterations, split_mode=mesh_split_mode,
            show_figures=show_figures, render=render
        )

        # ==== Calculate curvatures ====
        for cv_task in curv_calc_tasks:
            mesh_curv, full_C_gauss, full_C_mean = nemo_morph_curvature.main(
                img_path=path, t_select=t_select, c_select=c_select,
                num_samples=curv_num_samples, radius=cv_task["radius"], interp_k=curv_interp_k,
                mesh_name=cv_task["mesh_name"], flip_normals=cv_task["flip_normals"],
                show_figures=show_figures, render=render
            )

        # ==== Projection & nematic pipeline ====
        for p_task in proj_tasks:
            layer_label, layer_mesh, proj_layer = nemo_3_project_img.main(
                img_path=path, t_select=t_select, c_select=c_select,
                mesh_name=p_task["mesh_name"], dist_min=p_task["dist_min"], dist_max=p_task["dist_max"],
                dist_num=p_task["dist_num"], scale_down_mesh=proj_scale_down_mesh,
                proj_mode=p_task["proj_mode"], flip_normals=p_task["flip_normals"],
                show_figures=show_figures, render=render, correct_offset_automatic=False, initial_broad_scan=False
            )

            layer_label, layer_mesh, proj_layer, idxs_neigh, idxs_sel, directors_2dcurved = nemo_4_extract_nematic.main(
                img_path=path, t_select=t_select, c_select=c_select,
                layer_label=layer_label, patch_mode=nematic_patch_mode, patch_size=nematic_patch_size,
                compute_num=nematic_compute_num, normal_validity_k=nematic_normal_validity_k,
                normal_validity_thresh=nematic_normal_validity_thresh, grid_n_2dcurve_analysis=nematic_grid_n,
                debug_2dcurve_analysis=False, show_figures=show_figures, render=render
            )

            layer_label, layer_mesh, proj_layer, directors_2dcurved, nematic_avg_label, directors_2dcurved_avg, s_2dcurv = nemo_5_analyse_nematic.main(
                img_path=path, t_select=t_select, c_select=c_select,
                layer_label=layer_label, avg_mode=nematic_avg_mode, avg_size=nematic_avg_size
            )

            layer_label, layer_mesh, proj_layer, directors_2dcurved_avg, s_2dcurv, defect_coords = nemo_6_extract_defects.main(
                img_path=path, t_select=t_select, c_select=c_select,
                layer_label=layer_label, nematic_avg_label=nematic_avg_label,
                dist_cutoff_defect_localisation=defect_dist_cutoff_localisation,
                max_candidates_defect_localisation=defect_max_candidates,
                show_figures=show_figures, render=render
            )

            layer_label, layer_mesh, proj_layer, directors_2dcurved_avg, s_2dcurv, defect_idxs_calc, m_charge, charge_pol_linked_idxs, pol_vecfield = nemo_7_analyse_defects.main(
                img_path=path, t_select=t_select, c_select=c_select,
                layer_label=layer_label, nematic_avg_label=nematic_avg_label,
                topcurv_radius=defect_topcurv_radius, topcurv_interpk=defect_topcurv_interpk,
                topcharge_mode=defect_topcharge_mode, topcharge_size=defect_topcharge_size,
                pol_mode=defect_pol_mode, pol_size=defect_pol_size,
                show_figures=show_figures, render=render
            )

        # ==== Analyse inter-layer order ====
        for ilo_task in inter_layer_order_tasks:
            inter_layer_s_analysed = nemo_8_interlayer_nematic.main(
                img_path=path, t_select=t_select, c_select=c_select,
                layer_name_1=ilo_task["layer_name_1"], layer_name_2=ilo_task["layer_name_2"],
                patch_label_1=ilo_task["patch_label_1"], patch_label_2=ilo_task["patch_label_2"],
                show_figures=show_figures, render=render
            )

        # ==== Calculate thicknesses ====
        for th_task in thickness_calc_tasks:
            mesh_1, mesh_2, full_dist_vals = nemo_morph_thickness.main(
                img_path=path, t_select=t_select, c_select=c_select,
                mesh_1_name=th_task["mesh_1_name"], mesh_2_name=th_task["mesh_2_name"],
                thickness_sampl_number=thickness_num_samples, interp_k=thickness_interp_k,
                show_figures=show_figures, render=render
            )
